[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# FTS5Model and SearchField


## What you will be able to do

Build a full text index over a table you already have, with an `FTS5Model` and `SearchField`, and
check that the SQLite in front of you can do it at all before the first `CREATE VIRTUAL TABLE`.
Search it with `match`, rank the results, and join back to the real table to get the rows. Say why
handing a search box straight to `match` is handing your users a query language, and use `web_query`
so that a plus sign or a trailing `AND` is text rather than a syntax error. Keep a column in the
index without searching it. Say what a schema change to a virtual table costs.


## The idea

### The problem

`Book.blurb.contains("sea")` finds the rows whose blurb has that substring, in any order, with no
sense of which is more about the sea than the others, by reading every row. Full text search is a
different thing: the words are indexed, the match is by word rather than by substring, and the
results come back ranked.

SQLite does that with FTS5, and FTS5 takes a query language, not a phrase. `AND`, `OR`, `NOT`,
quoted phrases, `column:word`, parentheses and `*` all mean something. So the moment a search box is
wired to `match`, every visitor is typing into a parser. Somebody searches for `c++` and gets an
`OperationalError`, because `+` is syntax.

### What this notebook is about

FTS5 itself, the virtual table, the `MATCH` operator and bm25 ranking are SQLite's, and **sqlite3,
Deep Dive** is the guide that owns them. What is peewee's, and what this notebook teaches, is the
wrapper: an `FTS5Model` subclass, `SearchField`, the `content` option that points the index at a
real table, the `rowid` join back to it, and `web_query`, which turns what somebody typed into a
query FTS5 will accept.

### Why it works that way

A virtual table is not a table. It is a module with a table's interface, so the columns it declares
are the index's, not a copy of your data, and the usual SQL for changing a table does not apply to
it. With `content` pointing at a real table it stores only the index, and the text stays in one
place. That is also why a change to its columns is a drop and rebuild rather than an `ALTER`.

### Where this shows up

Any search box. The trap is not exotic: `c++`, a trailing `AND` from somebody who started typing a
second word, a stray hyphen, or a quote that was never closed. Each of those is a syntax error from
the database, arriving as a five hundred rather than as no results.

### What this notebook covers

Checking that FTS5 exists before relying on it. Building the index over a real table. Searching,
ranking, and joining back for the rows. `web_query` against the three things a search box will
receive. The unindexed column. What `ALTER` does to a virtual table. Then the four failures.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from peewee import CharField, Model, SqliteDatabase, TextField
from playhouse.sqlite_ext import FTS5Model, RowIDField, SearchField

db = SqliteDatabase(":memory:")


class Doc(Model):
    title = CharField()
    body = TextField()

    class Meta:
        database = db


class DocIndex(FTS5Model):
    rowid = RowIDField()
    title = SearchField()
    body = SearchField()

    class Meta:
        database = db
        options = {"content": Doc}


db.create_tables([Doc, DocIndex])
Doc.create(title="Learning C++", body="templates, memory, and the standard library")
DocIndex.rebuild()

for typed in ("c++", "memory AND"):
    try:
        found = [d.title for d in DocIndex.select().where(DocIndex.match(typed))]
        print(f"match({typed!r}) -> {found}")
    except Exception as error:
        print(f"match({typed!r}) -> {type(error).__name__}: {error}")
    safe = DocIndex.web_query(typed)
    print(f"  web_query gives {safe!r} -> "
          f"{[d.title for d in DocIndex.select().where(DocIndex.match(safe))]}")
```

```
match('c++') -> OperationalError: fts5: syntax error near "+"
  web_query gives '"c++"' -> ['Learning C++']
match('memory AND') -> OperationalError: fts5: syntax error near ""
  web_query gives '"memory"' -> ['Learning C++']
```

Two things a person would type, two errors, and the same two strings through `web_query` finding the
book. `web_query` quoted the first and dropped the dangling `AND` from the second. What it returns
is a string: a query FTS5 will accept, which still has to be handed to `match`.


## Setup

Six imports, peewee installed and pinned, a shelf of five books, and an index over them.

- `peewee` is the library, and `Model`, the field classes and `SqliteDatabase`, from it, are what a
  model is written with
- `FTS5Model`, `SearchField` and `RowIDField`, from `playhouse.sqlite_ext`, are the index
- `OperationalError` is what FTS5 raises at a query it cannot parse, caught several times below
- `subprocess`, `sys`, `version` and `PackageNotFoundError` install peewee 4.5.1 where the version is
  not that, as on Colab, whose 4.4.0 words some of these messages differently

The Setup cell checks for FTS5 before it does anything else, two ways: `FTS5Model.fts5_installed()`,
which peewee provides, and the compile option itself. FTS5 is a compile time option, and on a SQLite
built without it every cell below fails at the `CREATE VIRTUAL TABLE`. It is worth checking rather
than assuming, because the failure otherwise arrives as a table that cannot be made.

In 4.x these models bind to a plain `SqliteDatabase`. The separate `SqliteExtDatabase` that older
code uses for this is gone, its features having moved into the ordinary class.


In [1]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import CharField, Model, OperationalError, SqliteDatabase, TextField
from playhouse.sqlite_ext import FTS5Model, RowIDField, SearchField

SHELF = [                                                           # title, description, shelf mark
    ("The Sea, the Sea", "The sea, the sea, and a house above the sea. A sea story.", "A1"),
    ("A Quiet Engine", "Engines, quiet ones, and one mention of the sea.", "B2"),
    ("Tides", "The sea at night, tides, and the sea again by morning.", "A1"),
    ("Stone and Slate", "Granite, slate and quarries. No water at all.", "C3"),
    ("Learning C++", "Templates, memory and the standard library.", "D4"),
]

db = SqliteDatabase(":memory:", pragmas={"foreign_keys": 1})        # SQLite enforces nothing without this


print("peewee", peewee.__version__, "| FTS5 available:", FTS5Model.fts5_installed())
print("the compile option:",
      db.execute_sql("SELECT 1 FROM pragma_compile_options() "
                     "WHERE compile_options = 'ENABLE_FTS5'").fetchone() is not None)


class Book(Model):
    """The real table, where the text lives."""

    title = CharField(max_length=80)
    blurb = TextField()
    shelf = CharField(max_length=10)

    class Meta:
        database = db


class BookIndex(FTS5Model):
    """The virtual table that searches Book, holding no copy of the text itself."""

    rowid = RowIDField()
    title = SearchField()
    blurb = SearchField()
    shelf = SearchField(unindexed=True)

    class Meta:
        database = db
        options = {"content": Book}


db.create_tables([Book, BookIndex])
for title, blurb, shelf in SHELF:
    Book.create(title=title, blurb=blurb, shelf=shelf)
BookIndex.rebuild()

print("books:", Book.select().count(), "| indexed:", BookIndex.select().count())


peewee 4.5.1 | FTS5 available: True
the compile option: True
books: 5 | indexed: 5


## Worked examples

### The index is a table peewee did not write

`CREATE VIRTUAL TABLE` rather than `CREATE TABLE`, and the columns named are the index's:


In [2]:
print(BookIndex._schema._create_table().query()[0])
print()
print("Book columns:     ", [f.column_name for f in Book._meta.sorted_fields])
print("BookIndex columns:", [f.column_name for f in BookIndex._meta.sorted_fields])


CREATE VIRTUAL TABLE IF NOT EXISTS "bookindex" USING fts5 ("title", "blurb", "shelf" UNINDEXED, content="book")

Book columns:      ['id', 'title', 'blurb', 'shelf']
BookIndex columns: ['rowid', 'title', 'blurb', 'shelf']


`content="book"` is the part that matters. It tells FTS5 that the text lives in `book` and that the
virtual table should hold only the index, so nothing is stored twice. `rebuild()` is what fills the
index from the content table, and it is what has to be run after the rows change.

### Searching

`match` takes an FTS5 query and returns the index rows that match it:


In [3]:
found = BookIndex.select().where(BookIndex.match("sea"))
print("titles:", [row.title for row in found])
print()
print("by word, not by substring:")
for word in ("ea", "tide"):
    print(f"  match({word!r}):    ",
          [row.title for row in
           BookIndex.select().where(BookIndex.match(BookIndex.web_query(word)))])
    print(f"  contains({word!r}): ",
          [row.title for row in Book.select().where(Book.blurb.contains(word))])


titles: ['The Sea, the Sea', 'A Quiet Engine', 'Tides']

by word, not by substring:
  match('ea'):     []
  contains('ea'):  ['The Sea, the Sea', 'A Quiet Engine', 'Tides']
  match('tide'):     []
  contains('tide'):  ['Tides']


### Ranking

The order a search returns is not the order rows were written. `bm25` scores each match, and more
negative is more relevant, so ordering by it ascending puts the best first:


In [4]:
ranked = (BookIndex.select()
                   .where(BookIndex.match("sea"))
                   .order_by(BookIndex.bm25()))

for position, row in enumerate(ranked, start=1):
    times = row.blurb.lower().count("sea")
    print(f"  {position}. {row.title:<18} 'sea' appears {times} time{'' if times == 1 else 's'}")


  1. The Sea, the Sea   'sea' appears 4 times
  2. Tides              'sea' appears 2 times
  3. A Quiet Engine     'sea' appears 1 time


The counts are printed beside the order to show what the ranking is responding to. The score itself
is a float whose exact value depends on the SQLite build, so this notebook prints the order rather
than the numbers.

### Getting the real rows back

The index rows are not `Book` objects. `rowid` is the join back:


In [5]:
books = (Book.select()
             .join(BookIndex, on=(Book.id == BookIndex.rowid))
             .where(BookIndex.match(BookIndex.web_query("sea")))
             .order_by(BookIndex.bm25()))

print(" ".join(books.sql()[0].split())[:96], "...")
for book in books:
    print(f"  {book.title:<18} shelf {book.shelf}")


SELECT "t1"."id", "t1"."title", "t1"."blurb", "t1"."shelf" FROM "book" AS "t1" INNER JOIN "booki ...
  The Sea, the Sea   shelf A1
  Tides              shelf A1
  A Quiet Engine     shelf B2


Now they are `Book` instances, with every column the real table has, and the order is still the
search engine's.

### web_query, and what a search box actually contains

`web_query` translates what somebody typed into a query FTS5 will accept. It is a string in and a
string out, so the result still goes to `match`:


In [6]:
typed = ["sea", "c++", "sea AND", "sea -", '"the sea"', "sea OR stone", "sea -night", "title:tides"]

for entry in typed:
    print(f"  {entry!r:<16} -> {BookIndex.web_query(entry)!r}")


  'sea'            -> '"sea"'
  'c++'            -> '"c++"'
  'sea AND'        -> '"sea"'
  'sea -'          -> '"sea"'
  '"the sea"'      -> '"the sea"'
  'sea OR stone'   -> '"sea" OR "stone"'
  'sea -night'     -> '"sea" NOT "night"'
  'title:tides'    -> '"title": "tides"'


Quoting is what makes `c++` safe. The dangling `AND` and the stray hyphen are dropped rather than
passed through. The deliberate operators survive: `OR` stays, `-night` becomes `NOT "night"`, and
`title:tides` stays a column filter, so a person who knows the syntax still gets it.

All eight run:


In [7]:
for entry in typed:
    hits = [row.title for row in
            BookIndex.select().where(BookIndex.match(BookIndex.web_query(entry)))]
    print(f"  {entry!r:<16} -> {hits}")


  'sea'            -> ['The Sea, the Sea', 'A Quiet Engine', 'Tides']
  'c++'            -> ['Learning C++']
  'sea AND'        -> ['The Sea, the Sea', 'A Quiet Engine', 'Tides']
  'sea -'          -> ['The Sea, the Sea', 'A Quiet Engine', 'Tides']
  '"the sea"'      -> ['The Sea, the Sea', 'A Quiet Engine', 'Tides']
  'sea OR stone'   -> ['The Sea, the Sea', 'A Quiet Engine', 'Tides', 'Stone and Slate']
  'sea -night'     -> ['The Sea, the Sea', 'A Quiet Engine']
  'title:tides'    -> ['Tides']


### A column in the index that is not searched

`SearchField(unindexed=True)` stores a column in the virtual table without indexing its words, which
is how a shelf mark or an identifier rides along without becoming searchable text:


In [8]:
print("in the DDL:", "UNINDEXED" in BookIndex._schema._create_table().query()[0])
print("readable:", [(row.title, row.shelf) for row in BookIndex.select().limit(2)])
print("searchable:", [row.title for row in
                      BookIndex.select().where(BookIndex.match(BookIndex.web_query("A1")))])


in the DDL: True
readable: [('The Sea, the Sea', 'A1'), ('A Quiet Engine', 'B2')]
searchable: []


The shelf mark is there to read and finds nothing when searched for, which is the point: without
`unindexed`, every shelf mark would be a word in the index and a search for `A1` would return books
by their location rather than their subject.

### A virtual table cannot be altered

The **Migrations** notebook is about changing a schema. This is the one table that will not be
changed:


In [9]:
try:
    db.execute_sql('ALTER TABLE "bookindex" ADD COLUMN "author" TEXT')
except OperationalError as error:
    print("peewee.OperationalError:", error)

print()
print("the way to change it:")
print("  BookIndex.drop_table(), a new class, create_tables([...]), BookIndex.rebuild()")


peewee.OperationalError: virtual tables may not be altered

the way to change it:
  BookIndex.drop_table(), a new class, create_tables([...]), BookIndex.rebuild()


Adding a searchable column means dropping the virtual table and building it again from the content
table, which is cheap when the content table is the source of truth and expensive when it is not.
It is one more reason to keep the text in a real table and the index over it.

### When to reach for which

| What you want | How to write it |
|---|---|
| a substring, anywhere, unranked | `Model.column.contains("sea")` |
| words, indexed and ranked | an `FTS5Model` with `SearchField` columns |
| the index to hold no copy of the text | `options = {"content": RealModel}` |
| to fill or refill the index | `Index.rebuild()` |
| a search from your own code | `Index.match("sea OR stone")` |
| a search from a person's typing | `Index.match(Index.web_query(typed))` |
| best matches first | `.order_by(Index.bm25())` |
| the real rows | `.join(Index, on=(Real.id == Index.rowid))` |
| a column stored but not searched | `SearchField(unindexed=True)` |
| to change the index's columns | drop it, redeclare it, create it, rebuild it |

`web_query` is the default for anything a person typed, and `match` is for queries your own code
built. The distinction is the whole notebook: one of them is a query language and the other is a
search box.

### A search endpoint, finished

What all of it is for: a function that takes whatever was typed, never raises on it, ranks the
results and returns real rows.


In [10]:
def search(typed, limit=5):
    """Search the shelf for whatever somebody typed, best first."""
    if not typed.strip():
        return []
    query = BookIndex.web_query(typed)
    if not query:                                                   # nothing searchable was typed
        return []
    return list(Book.select()
                    .join(BookIndex, on=(Book.id == BookIndex.rowid))
                    .where(BookIndex.match(query))
                    .order_by(BookIndex.bm25())
                    .limit(limit))


for entry in ("sea", "c++", "sea AND", "  ", "-", "granite OR templates"):
    print(f"  {entry!r:<22} {[book.title for book in search(entry)]}")


  'sea'                  ['The Sea, the Sea', 'Tides', 'A Quiet Engine']
  'c++'                  ['Learning C++']
  'sea AND'              ['The Sea, the Sea', 'Tides', 'A Quiet Engine']
  '  '                   []
  '-'                    []
  'granite OR templates' ['Learning C++', 'Stone and Slate']


Six things a person might type, including two that are not searches at all, and not one of them
raises. The empty results are empty because nothing matched or nothing was asked, which is a result
a page can render.

### Where each part came from

| In the endpoint | What it relies on | The section that showed it |
|---|---|---|
| `BookIndex.web_query(typed)` | a query FTS5 will accept | web_query |
| `BookIndex.match(query)` | the search itself | Searching |
| `.join(BookIndex, on=(Book.id == BookIndex.rowid))` | real rows from index matches | Getting the real rows back |
| `.order_by(BookIndex.bm25())` | best matches first | Ranking |
| the empty string check | `web_query` can return nothing to search for | web_query |
| `options = {"content": Book}` | one copy of the text | The index is a table peewee did not write |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/09-fts5model-and-searchfield-solutions.ipynb).

**1.** Search the index for `stone` and print the titles, then do the same with `contains` on the
real table and say what is different about the two answers.


In [11]:
# your code here


**2.** Print what `web_query` makes of `tides OR granite`, `"the sea"` and `sea AND OR`, and run
each of the three.


In [12]:
# your code here


**3.** Search for `sea`, ranked, and return `Book` rows rather than index rows, printing each book's
shelf mark.


In [13]:
# your code here


**4.** Add a book to the real table, show that the index does not find it, then make the index find
it.


In [14]:
# your code here


**5.** Show one search that raises through `match` and does not raise through `web_query`, and print
both outcomes.


In [15]:
# your code here


**6.** Write a `safe_search` that returns an empty list rather than raising, whatever it is given,
and try it on four different inputs.


In [16]:
# your code here


## Common errors

### peewee.OperationalError: fts5: syntax error near "+"


In [17]:
BookIndex.select().where(BookIndex.match("c++")).count()


OperationalError: fts5: syntax error near "+"

`+` is not a word character to FTS5, and a bare `+` is not valid syntax either. The same is true of
anything else the query language reserves, which is why no list of characters to strip is ever
complete.

Quoting is the fix, and `web_query` is quoting done for you:


In [18]:
print("web_query:", repr(BookIndex.web_query("c++")))
print("found:", [row.title for row in
                 BookIndex.select().where(BookIndex.match(BookIndex.web_query("c++")))])
print("by hand:", [row.title for row in BookIndex.select().where(BookIndex.match('"c++"'))])


web_query: '"c++"'
found: ['Learning C++']
by hand: ['Learning C++']


### peewee.OperationalError: fts5: syntax error near ""


In [19]:
BookIndex.select().where(BookIndex.match("sea AND")).count()


OperationalError: fts5: syntax error near ""

An operator with nothing after it. The message names the empty string because that is what FTS5
found where it expected a term, and a trailing hyphen gives the same message. An unclosed quote is a
different message again, which is the point: there is no one error to catch.

This is what makes it a real failure rather than a curiosity. A search box sends a request on every
submit, and `sea AND` is what `sea AND stone` looks like a moment before it is finished.


In [20]:
for half_typed in ("sea AND", "sea -", 'sea "night'):
    try:
        BookIndex.select().where(BookIndex.match(half_typed)).count()
        print(f"  {half_typed!r:<14} no error")
    except OperationalError as error:
        print(f"  {half_typed!r:<14} {error}")
    print(f"  {'':<14} web_query -> {BookIndex.web_query(half_typed)!r}")


  'sea AND'      fts5: syntax error near ""
                 web_query -> '"sea"'
  'sea -'        fts5: syntax error near ""
                 web_query -> '"sea"'
  'sea "night'   unterminated string
                 web_query -> '"sea" AND "night"'


### No error, and a book that is not found: rows added after the index was built


In [21]:
Book.create(title="The Tide Tables", blurb="Tables of the tides, sea by sea.", shelf="A2")

print("in the real table:", Book.select().where(Book.title == "The Tide Tables").count())
print("found by a search:", [row.title for row in
                             BookIndex.select().where(BookIndex.match(BookIndex.web_query("tide")))])


in the real table: 1
found by a search: []


The row is in `book` and the index has not heard of it. With `content` pointing at a real table, the
virtual table is a snapshot of the words as of the last `rebuild`, and writing to the real table does
not touch it.

Nothing raises, because there is nothing wrong: a search found what the index holds. `rebuild` is one
answer, and for a table of any size the other is a trigger, or an application that writes to both:


In [22]:
BookIndex.rebuild()
print("after rebuild:", [row.title for row in
                         BookIndex.select().where(BookIndex.match(BookIndex.web_query("tide")))])


after rebuild: ['The Tide Tables']


### peewee.OperationalError: virtual tables may not be altered


In [23]:
db.execute_sql('ALTER TABLE "bookindex" ADD COLUMN "author" TEXT')


OperationalError: virtual tables may not be altered

Every migration tool in the **Migrations** notebook writes `ALTER TABLE`, and none of them can
change this one. A virtual table's columns are part of how the module was created, so adding one
means creating the table again.

The drop and rebuild, which is cheap here because the text lives in `book` and the index holds none
of it:


In [24]:
BookIndex.drop_table()
db.create_tables([BookIndex])
BookIndex.rebuild()

print("rebuilt, indexed rows:", BookIndex.select().count())
print("still searchable:", [row.title for row in
                            BookIndex.select().where(BookIndex.match(BookIndex.web_query("sea")))])


rebuilt, indexed rows: 6
still searchable: ['The Sea, the Sea', 'A Quiet Engine', 'Tides', 'The Tide Tables']


## Recap

- FTS5 is a compile time option. `FTS5Model.fts5_installed()` answers whether it is there, and it is
  worth asking before the first `CREATE VIRTUAL TABLE`.
- An `FTS5Model` with `SearchField` columns is a virtual table. With `options = {"content": Model}`
  it holds only the index, and `rebuild()` fills it from the real table.
- `match` takes an FTS5 query, which is a language: `AND`, `OR`, `NOT`, quotes, `column:word` and
  parentheses all mean something, and `+` is a syntax error.
- `web_query` translates what a person typed into a query FTS5 accepts, quoting what is not an
  operator and dropping what is dangling. It returns a string, which still goes to `match`.
- `bm25` ranks the matches, most relevant first when ordered ascending.
- The index rows are not your rows. `rowid` joins back to the content table.
- `SearchField(unindexed=True)` stores a column without making its words searchable.
- A virtual table cannot be altered. Changing its columns is a drop, a redeclare and a rebuild.
- In 4.x all of this binds to a plain `SqliteDatabase`; `SqliteExtDatabase` is gone.


## What is next

The **Migrations** notebook is about changing a schema that already has data in it: `pwmigrate`, the
difference between what the models say and what the database holds, and the `down` step that was
written and never run.


---

&#8592; **Previous:** [JSON Columns](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/08-json-columns.ipynb)  &nbsp;·&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Migrations](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/10-migrations.ipynb) &#8594;
